# AMAG Architecture Experiments

Three controlled experiments testing architectural improvements to the AMAG model (NeurIPS 2023) for monkey neural activity forecasting.

## Experiments

| # | Name | Change | Hypothesis |
|---|------|--------|------------|
| 1 | **Multi-Hop SI** | Run spatial interaction twice (2-hop propagation) | Affi (multi-region) benefits more than Beignet (single M1) |
| 2 | **Delta TR** | Feed spatial correction (z-h) to TR, init hidden state from TE | Removes redundant temporal re-encoding in TR |
| 3 | **Interleaved TE-SI + Aux Loss** | Interleave temporal and spatial processing; add next-step prediction loss | Gives TE spatial awareness; provides per-timestep SI training signal |

All experiments train on both Affi (239ch) and Beignet (89ch) with identical hyperparameters.

## Setup Instructions

1. **Enable GPU**: Runtime > Change runtime type > T4 GPU
2. **Upload data to Google Drive**: Create `MonkeyNeural/dataset/` and upload the 5 `.npz` files
3. **Upload code to Google Drive**: Upload `src/` and `replication/` folders to `MonkeyNeural/`
4. **Run all cells**: Runtime > Run all (estimated ~10 hours total on T4)

Results are saved to Google Drive automatically. The notebook can be resumed after disconnection.

In [ ]:
# ── STEP 1: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
print('Drive mounted. Contents of MonkeyNeural/:',
      os.listdir('/content/drive/MyDrive/MonkeyNeural')
      if os.path.exists('/content/drive/MyDrive/MonkeyNeural')
      else 'NOT FOUND -- create the folder first')

In [ ]:
# ── STEP 2: Extract project code ────────────────────────────────────────────
import shutil

DRIVE_PROJECT = '/content/drive/MyDrive/MonkeyNeural'
LOCAL_PROJECT = '/content/MonkeyNeuralForecastingClaude'

os.makedirs(LOCAL_PROJECT, exist_ok=True)

# Copy src/ and replication/ from Drive to local (faster execution)
for folder in ['src', 'replication']:
    src_path = os.path.join(DRIVE_PROJECT, folder)
    dst_path = os.path.join(LOCAL_PROJECT, folder)
    if os.path.exists(src_path):
        if os.path.exists(dst_path):
            shutil.rmtree(dst_path)
        shutil.copytree(src_path, dst_path)
        print(f'Copied {folder}/ to {dst_path}')
    else:
        print(f'WARNING: {src_path} not found in Drive')

# Symlink dataset/ to avoid copying 400MB+
dataset_link = os.path.join(LOCAL_PROJECT, 'dataset')
dataset_src = os.path.join(DRIVE_PROJECT, 'dataset')
if not os.path.exists(dataset_link) and os.path.exists(dataset_src):
    os.symlink(dataset_src, dataset_link)
    print(f'Symlinked dataset/ -> {dataset_src}')

print('\nProject structure:')
for item in sorted(os.listdir(LOCAL_PROJECT)):
    print(f'  {item}/')

In [ ]:
# ── STEP 3: Install dependencies ────────────────────────────────────────────
!pip install -q pyyaml einops
print('Dependencies ready')

In [ ]:
# ── STEP 4: Add project to Python path + verify GPU ─────────────────────────
import sys
LOCAL_PROJECT = '/content/MonkeyNeuralForecastingClaude'
if LOCAL_PROJECT not in sys.path:
    sys.path.insert(0, LOCAL_PROJECT)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── STEP 5: Quick data check ─────────────────────────────────────────────────
import numpy as np

DATASET_DIR = os.path.join(LOCAL_PROJECT, 'dataset')
for fname in ['train_data_affi.npz', 'train_data_beignet.npz']:
    path = os.path.join(DATASET_DIR, fname)
    if os.path.exists(path):
        d = np.load(path, allow_pickle=True)
        key = 'data' if 'data' in d else list(d.keys())[0]
        arr = d[key]
        print(f'{fname}: shape={arr.shape}, dtype={arr.dtype}')
    else:
        print(f'MISSING: {path}')

---
# Section 1: Shared Infrastructure

Imports, constants, model components shared across all experiments, training loop, and evaluation helpers.

In [ ]:
# ── Imports and Constants ─────────────────────────────────────────────────────
import os, sys, json, time, random, shutil, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from typing import Optional, Dict, Tuple, List
import matplotlib.pyplot as plt

LOCAL_PROJECT = '/content/MonkeyNeuralForecastingClaude'
if LOCAL_PROJECT not in sys.path:
    sys.path.insert(0, LOCAL_PROJECT)

# Project imports (unchanged infrastructure)
from src.data.dataset import (
    get_dataloaders, MONKEY_CHANNELS, MONKEY_FILES, DATASET_DIR, load_npz
)
from src.evaluation.evaluator import evaluate_model, print_results
from src.models.base import ForecastingModel
from src.models.amag.components import TemporalEncoder, TemporalReadout

# ── Constants (identical for all experiments) ──
SEED = 42
MONKEYS = ['beignet', 'affi']  # beignet first (faster, validates code)
HIDDEN_SIZE = 64
N_FEATURES = 9
N_PRED_STEPS = 10
N_CTX_STEPS = 10
BATCH_SIZE = 32

DRIVE_RESULTS = '/content/drive/MyDrive/MonkeyNeural/experiment_results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

TRAINING_CONFIG = {
    'lr': 5e-4,
    'weight_decay': 1e-5,
    'n_epochs': 500,
    'lr_decay': 0.95,
    'lr_decay_every': 50,
    'patience': 100,
    'seed': SEED,
    'log_every': 25,
}

def detect_device():
    if torch.cuda.is_available():
        return 'cuda'
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'

DEVICE = detect_device()
print(f'Device: {DEVICE}')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def compute_correlation_matrix(monkey):
    """Compute LMP channel-channel correlation from training data context steps."""
    n_channels = MONKEY_CHANNELS[monkey]
    train_path = os.path.join(DATASET_DIR, MONKEY_FILES[monkey]['train'])
    raw = load_npz(train_path)
    lmp = raw[:, :10, :, 0].reshape(-1, n_channels)
    lmp_centered = lmp - lmp.mean(axis=0)
    norm = np.sqrt((lmp_centered ** 2).sum(axis=0) + 1e-12)
    lmp_normed = lmp_centered / norm
    corr = (lmp_normed.T @ lmp_normed) / len(lmp)
    return corr.astype(np.float32)

print('Infrastructure loaded.')

In [ ]:
# ── Spatial Interaction Module (shared by all experiments) ─────────────────────
# Copied from replication/amag/model.py with full ablation support.

class SpatialInteractionAblatable(nn.Module):
    """
    AMAG Spatial Interaction module with ablation flags.

    Computes: z^(v)_t = b1*h^(v)_t + b2*FC(a^(v)_t) + b3*FC(m^(v)_t)

    Where:
      a^(v)_t = sum_u S(u,v) * A_a(u,v) * h^(u)_t   [Add: linear spatial summation]
      m^(v)_t = sum_u A_m(u,v) * (h^(u)_t . h^(v)_t) [Mod: multiplicative gain gating]
      S(u,v)  = sigmoid(MLP([H^u, H^v]))              [Adaptor: sample-dependent scaling]
    """

    def __init__(self, hidden_size, n_channels, init_corr=None,
                 use_add=True, use_mul=True, use_adaptor=True, learnable_adj=True):
        super().__init__()
        self.hidden_size = hidden_size
        self.n_channels = n_channels
        self.use_add = use_add
        self.use_mul = use_mul
        self.use_adaptor = use_adaptor

        if init_corr is not None:
            corr_tensor = torch.tensor(init_corr, dtype=torch.float32)
        else:
            std = 1.0 / (n_channels ** 0.5)
            corr_tensor = torch.randn(n_channels, n_channels) * std

        if learnable_adj:
            self.A_a = nn.Parameter(corr_tensor.clone())
            self.A_m = nn.Parameter(corr_tensor.clone())
        else:
            self.register_buffer('A_a', corr_tensor.clone())
            self.register_buffer('A_m', corr_tensor.clone())

        self.beta = nn.Parameter(torch.ones(3))

        if use_add:
            self.fc_add = nn.Linear(hidden_size, hidden_size, bias=False)
        if use_mul:
            self.fc_mod = nn.Linear(hidden_size, hidden_size, bias=False)

        if use_add and use_adaptor:
            self.adaptor_mlp = nn.Sequential(
                nn.Linear(hidden_size * 2, hidden_size),
                nn.ReLU(),
                nn.Linear(hidden_size, 1),
            )

    def forward(self, H):
        B, T, C, d = H.shape
        n_active = 1 + int(self.use_add) + int(self.use_mul)
        beta = F.softmax(self.beta[:n_active], dim=0) * n_active

        Z_list = []
        for t in range(T):
            h_t = H[:, t, :, :]  # (B, C, d)
            components = [h_t]

            if self.use_add:
                A_a_norm = torch.tanh(self.A_a)
                if self.use_adaptor:
                    H_mean = H[:, :t + 1, :, :].mean(dim=1)
                    h_u = H_mean.unsqueeze(2).expand(-1, -1, C, -1)
                    h_v = H_mean.unsqueeze(1).expand(-1, C, -1, -1)
                    h_uv = torch.cat([h_u, h_v], dim=-1)
                    S = torch.sigmoid(self.adaptor_mlp(h_uv).squeeze(-1))
                    A_a_scaled = A_a_norm.unsqueeze(0) * S
                    a_t = torch.einsum('bvu, bud -> bvd', A_a_scaled, h_t)
                else:
                    a_t = torch.einsum('vu, bud -> bvd', A_a_norm, h_t)
                components.append(self.fc_add(a_t))

            if self.use_mul:
                A_m_norm = torch.tanh(self.A_m)
                h_u = h_t.unsqueeze(1).expand(-1, C, -1, -1)
                h_v = h_t.unsqueeze(2).expand(-1, -1, C, -1)
                hadamard = h_u * h_v
                m_t = torch.einsum('vu, bvud -> bvd', A_m_norm, hadamard)
                components.append(self.fc_mod(m_t))

            z_t = sum(beta[i] * components[i] for i in range(len(components)))
            Z_list.append(z_t)

        return torch.stack(Z_list, dim=1)

print('SpatialInteractionAblatable defined.')

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────────

class StandardTrainer:
    """Standard MSE trainer with early stopping and checkpointing."""

    def __init__(self, model, train_loader, val_loader, config, **kwargs):
        self.config = config
        self.device = torch.device(config.get('device', DEVICE))
        set_seed(config.get('seed', SEED))
        self.model = model.to(self.device)
        self.train_loader = train_loader
        self.val_loader = val_loader

        self.optimizer = torch.optim.Adam(
            model.parameters(),
            lr=config.get('lr', 5e-4),
            weight_decay=config.get('weight_decay', 1e-5),
        )
        self.criterion = nn.MSELoss()
        self.scheduler = torch.optim.lr_scheduler.StepLR(
            self.optimizer,
            step_size=config.get('lr_decay_every', 50),
            gamma=config.get('lr_decay', 0.95),
        )
        self.checkpoint_dir = config.get('checkpoint_dir', '/tmp/ckpt')
        os.makedirs(self.checkpoint_dir, exist_ok=True)
        self.history = []
        self.best_val_mse = float('inf')
        self.patience_counter = 0

    def _train_epoch(self):
        self.model.train()
        total_loss, n = 0.0, 0
        for X, Y in self.train_loader:
            X, Y = X.to(self.device), Y.to(self.device)
            self.optimizer.zero_grad()
            pred = self.model(X)
            loss = self.criterion(pred, Y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()
            total_loss += loss.item()
            n += 1
        return total_loss / max(n, 1)

    def _val_epoch(self):
        self.model.eval()
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for X, Y in self.val_loader:
                X, Y = X.to(self.device), Y.to(self.device)
                pred = self.model(X)
                loss = self.criterion(pred, Y)
                total_loss += loss.item()
                n += 1
        return total_loss / max(n, 1)

    def save_checkpoint(self, tag='best'):
        path = os.path.join(self.checkpoint_dir, f'{tag}.pth')
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'best_val_mse': self.best_val_mse,
            'norm_stats': self.config.get('norm_stats', None),
        }, path)

    def load_best_checkpoint(self):
        path = os.path.join(self.checkpoint_dir, 'best.pth')
        state = torch.load(path, weights_only=False, map_location=self.device)
        self.model.load_state_dict(state['model_state_dict'])

    def train(self, verbose=True):
        n_epochs = self.config.get('n_epochs', 500)
        patience = self.config.get('patience', 100)
        log_every = self.config.get('log_every', 25)

        for epoch in range(1, n_epochs + 1):
            t0 = time.time()
            train_mse = self._train_epoch()
            val_mse = self._val_epoch()
            self.scheduler.step()

            improved = val_mse < self.best_val_mse
            if improved:
                self.best_val_mse = val_mse
                self.patience_counter = 0
                self.save_checkpoint('best')
            else:
                self.patience_counter += 1

            self.history.append({
                'epoch': epoch, 'train_mse': train_mse, 'val_mse': val_mse,
                'best_val_mse': self.best_val_mse, 'time_s': round(time.time() - t0, 2),
            })

            if verbose and (epoch % log_every == 0 or epoch == 1):
                mark = '*' if improved else ' '
                print(f'  Ep {epoch:4d} {mark} | train={train_mse:.6f} val={val_mse:.6f} '
                      f'best={self.best_val_mse:.6f} | {time.time() - t0:.1f}s')

            if self.patience_counter >= patience:
                if verbose:
                    print(f'  Early stop at epoch {epoch}')
                break

        if verbose:
            print(f'  Done. Best val MSE: {self.best_val_mse:.6f}')
        return {'best_val_mse': self.best_val_mse, 'history': self.history}


class AuxLossTrainer(StandardTrainer):
    """Trainer for models returning (pred, aux_loss) during training."""

    def __init__(self, model, train_loader, val_loader, config, aux_weight=0.1):
        super().__init__(model, train_loader, val_loader, config)
        self.aux_weight = aux_weight

    def _train_epoch(self):
        self.model.train()
        total_loss, n = 0.0, 0
        for X, Y in self.train_loader:
            X, Y = X.to(self.device), Y.to(self.device)
            self.optimizer.zero_grad()
            output = self.model(X)
            if isinstance(output, tuple):
                pred, aux_loss = output
            else:
                pred, aux_loss = output, torch.tensor(0.0, device=self.device)
            forecast_loss = self.criterion(pred, Y)
            loss = forecast_loss + self.aux_weight * aux_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()
            total_loss += forecast_loss.item()  # log forecast loss only for fair comparison
            n += 1
        return total_loss / max(n, 1)

    def _val_epoch(self):
        self.model.eval()
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for X, Y in self.val_loader:
                X, Y = X.to(self.device), Y.to(self.device)
                output = self.model(X)
                pred = output[0] if isinstance(output, tuple) else output
                loss = self.criterion(pred, Y)
                total_loss += loss.item()
                n += 1
        return total_loss / max(n, 1)


print('Trainers defined.')

In [ ]:
# ── Experiment Orchestrator and Comparison Helpers ─────────────────────────────

ALL_RESULTS = {}  # key: (experiment_name, monkey) -> results dict


def run_experiment(experiment_name, model_factory, monkeys=None,
                   trainer_class=StandardTrainer, trainer_kwargs=None,
                   training_config=None, batch_size=BATCH_SIZE):
    """
    Train and evaluate a model on specified monkeys.

    Args:
        experiment_name: string identifier for results tracking
        model_factory: callable(monkey, init_corr) -> nn.Module
        monkeys: list of monkey names (default: MONKEYS)
        trainer_class: StandardTrainer or AuxLossTrainer
        trainer_kwargs: extra kwargs for trainer constructor
        training_config: override TRAINING_CONFIG
        batch_size: override BATCH_SIZE (for OOM fallback)
    """
    if monkeys is None:
        monkeys = MONKEYS
    if training_config is None:
        training_config = TRAINING_CONFIG.copy()
    if trainer_kwargs is None:
        trainer_kwargs = {}

    results = {}
    for monkey in monkeys:
        print(f'\n{"="*60}')
        print(f'  {experiment_name} -- {monkey.upper()}')
        print(f'{"="*60}')

        # Check for existing checkpoint (resume support)
        ckpt_dir = os.path.join(DRIVE_RESULTS, experiment_name, monkey, 'checkpoints')
        ckpt_path = os.path.join(ckpt_dir, 'best.pth')
        if os.path.exists(ckpt_path):
            print(f'  Found existing checkpoint at {ckpt_path}')
            print(f'  Loading and evaluating (skip training)...')
            set_seed(SEED)
            train_loader, val_loader, stats = get_dataloaders(
                monkey, batch_size=batch_size, val_fraction=0.15, seed=SEED)
            init_corr = compute_correlation_matrix(monkey)
            model = model_factory(monkey, init_corr)
            state = torch.load(ckpt_path, weights_only=False, map_location=DEVICE)
            model.load_state_dict(state['model_state_dict'])
            model = model.to(DEVICE)
            eval_results = evaluate_model(model, val_loader)
            print_results(eval_results, prefix=f'{experiment_name}/{monkey}')
            info = model.get_model_info()
            results[monkey] = {
                'mse': eval_results['mse'],
                'r2': eval_results['r2']['mean'],
                'corr': eval_results['correlation']['mean'],
                'per_timestep_mse': eval_results['per_timestep_mse'].tolist(),
                'per_channel_mse': eval_results['per_channel_mse'].tolist(),
                'best_epoch_mse': state.get('best_val_mse', eval_results['mse']),
                'history': [],
                'n_params': info['n_params'],
            }
            ALL_RESULTS[(experiment_name, monkey)] = results[monkey]
            continue

        # Train from scratch
        set_seed(SEED)
        train_loader, val_loader, stats = get_dataloaders(
            monkey, batch_size=batch_size, val_fraction=0.15, seed=SEED)
        init_corr = compute_correlation_matrix(monkey)

        model = model_factory(monkey, init_corr)
        info = model.get_model_info()
        print(f'  Params: {info["n_params"]:,} ({info["n_params_M"]}M)')

        cfg = {**training_config, 'device': DEVICE,
               'checkpoint_dir': ckpt_dir, 'norm_stats': stats}

        trainer = trainer_class(model, train_loader, val_loader, cfg, **trainer_kwargs)
        train_result = trainer.train()
        trainer.load_best_checkpoint()

        eval_results = evaluate_model(model, val_loader)
        print_results(eval_results, prefix=f'{experiment_name}/{monkey}')

        # Save training log to Drive
        log_dir = os.path.join(DRIVE_RESULTS, experiment_name, monkey)
        os.makedirs(log_dir, exist_ok=True)
        with open(os.path.join(log_dir, 'training_log.json'), 'w') as f:
            json.dump({'history': train_result['history'],
                       'best_val_mse': train_result['best_val_mse']}, f, indent=2)

        results[monkey] = {
            'mse': eval_results['mse'],
            'r2': eval_results['r2']['mean'],
            'corr': eval_results['correlation']['mean'],
            'per_timestep_mse': eval_results['per_timestep_mse'].tolist(),
            'per_channel_mse': eval_results['per_channel_mse'].tolist(),
            'best_epoch_mse': train_result['best_val_mse'],
            'history': train_result['history'],
            'n_params': info['n_params'],
        }
        ALL_RESULTS[(experiment_name, monkey)] = results[monkey]

    return results


def print_comparison_table(experiment_names, title='Comparison'):
    """Print a formatted comparison table."""
    print(f'\n{"="*85}')
    print(f'  {title}')
    print(f'{"="*85}')
    print(f'  {"Experiment":<28} {"Monkey":<10} {"Val MSE":<12} {"R2":<8} {"Corr":<8} {"Params":<10}')
    print(f'  {"-"*80}')
    for name in experiment_names:
        for monkey in MONKEYS:
            key = (name, monkey)
            if key in ALL_RESULTS:
                r = ALL_RESULTS[key]
                print(f'  {name:<28} {monkey:<10} {r["mse"]:<12.6f} '
                      f'{r["r2"]:<8.4f} {r["corr"]:<8.4f} {r["n_params"]:<10,}')

    # Delta vs baseline
    if 'baseline' in experiment_names:
        print(f'\n  Delta vs Baseline:')
        for name in experiment_names:
            if name == 'baseline':
                continue
            for monkey in MONKEYS:
                base = ALL_RESULTS.get(('baseline', monkey))
                exp = ALL_RESULTS.get((name, monkey))
                if base and exp:
                    delta = exp['mse'] - base['mse']
                    pct = delta / base['mse'] * 100
                    direction = 'BETTER' if delta < 0 else 'worse'
                    print(f'    {name}/{monkey}: {delta:+.6f} ({pct:+.2f}%) {direction}')


def plot_experiment_comparison(experiment_names, title_suffix=''):
    """Plot per-timestep MSE and learning curves for experiments."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for idx, monkey in enumerate(MONKEYS):
        ax = axes[idx]
        for name in experiment_names:
            key = (name, monkey)
            if key in ALL_RESULTS:
                ts_mse = ALL_RESULTS[key]['per_timestep_mse']
                ax.plot(range(1, len(ts_mse) + 1), ts_mse, marker='o', label=name)
        ax.set_xlabel('Prediction Timestep')
        ax.set_ylabel('MSE')
        ax.set_title(f'{monkey.capitalize()} -- Per-Timestep MSE {title_suffix}')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = os.path.join(DRIVE_RESULTS, f'per_timestep_mse{title_suffix.replace(" ", "_")}.png')
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f'Saved: {save_path}')


def plot_learning_curves(experiment_names, title_suffix=''):
    """Plot training and validation learning curves."""
    n_monkeys = len(MONKEYS)
    fig, axes = plt.subplots(1, n_monkeys, figsize=(14, 5))
    if n_monkeys == 1:
        axes = [axes]

    for idx, monkey in enumerate(MONKEYS):
        ax = axes[idx]
        for name in experiment_names:
            key = (name, monkey)
            if key in ALL_RESULTS and ALL_RESULTS[key]['history']:
                history = ALL_RESULTS[key]['history']
                epochs = [h['epoch'] for h in history]
                val_mse = [h['val_mse'] for h in history]
                ax.plot(epochs, val_mse, label=f'{name} (val)', alpha=0.8)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Val MSE')
        ax.set_title(f'{monkey.capitalize()} -- Learning Curves {title_suffix}')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = os.path.join(DRIVE_RESULTS, f'learning_curves{title_suffix.replace(" ", "_")}.png')
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f'Saved: {save_path}')


print('Orchestrator and plotting helpers defined.')

---
# Section 2: Baseline AMAG (Control Run)

Standard AMAG architecture: TE (per-channel GRU) -> SI (additive + multiplicative graph message passing) -> TR (per-channel GRU -> linear).

This is the control run with identical hyperparameters, seed, and data split as all experiments.

In [ ]:
# ── Baseline AMAG Model ───────────────────────────────────────────────────────

class AMAGBaseline(ForecastingModel):
    """Standard AMAG: TE -> SI -> TR."""

    def __init__(self, n_channels, n_features=9, hidden_size=64,
                 n_pred_steps=10, init_corr=None):
        super().__init__()
        self.n_channels = n_channels
        self.hidden_size = hidden_size

        self.te = TemporalEncoder(n_features, hidden_size)
        self.si = SpatialInteractionAblatable(
            hidden_size=hidden_size, n_channels=n_channels,
            init_corr=init_corr, use_add=True, use_mul=True,
            use_adaptor=True, learnable_adj=True)
        self.tr = TemporalReadout(hidden_size, n_pred_steps)

    def forward(self, x):
        H = self.te(x)    # (B, T, C, d)
        Z = self.si(H)    # (B, T, C, d)
        return self.tr(Z) # (B, T_pred, C)


def make_baseline_amag(monkey, init_corr):
    return AMAGBaseline(
        n_channels=MONKEY_CHANNELS[monkey],
        n_features=N_FEATURES,
        hidden_size=HIDDEN_SIZE,
        n_pred_steps=N_PRED_STEPS,
        init_corr=init_corr,
    )


print('AMAGBaseline defined.')

In [ ]:
# ── Train Baseline ────────────────────────────────────────────────────────────
baseline_results = run_experiment('baseline', make_baseline_amag)
print_comparison_table(['baseline'], title='Baseline Results')

---
# Section 3: Experiment 1 -- Multi-Hop Spatial Interaction

## Problem

The current SI module performs **single-hop** message passing. At each timestep, channel v receives information only from its direct neighbors. Information from distant brain regions (e.g., DLPFC) cannot reach M1 through intermediate relays (DLPFC -> PM -> M1) in a single pass.

In standard GNNs, multi-hop propagation is achieved by stacking multiple message-passing layers. This allows the receptive field to grow, so that signals from 2-hop neighbors (neighbors of neighbors) can influence the target node.

## Change

Run the SI module **twice** (weight-tied): first pass produces Z1 from H, second pass produces Z2 from Z1. A learnable gate `alpha` blends the two hops: `Z = (1-alpha)*Z1 + alpha*Z2`. This adds only **1 parameter** (the gate scalar).

## Hypothesis

- **Affi** (5 cortical regions, 239 channels) should benefit more because inter-regional relay paths exist (DLPFC -> PM -> M1)
- **Beignet** (single M1 region, 89 channels) has less inter-regional structure, so the second hop should add little
- If `alpha` converges near 0, the second hop is learned to be unnecessary

In [ ]:
# ── Experiment 1: Multi-Hop SI Model ──────────────────────────────────────────

class AMAG_MultiHop(ForecastingModel):
    """
    AMAG with 2-hop spatial interaction.
    Forward: H -> Z1 = SI(H) -> Z2 = SI(Z1) -> Z = blend(Z1, Z2) -> TR(Z)
    Weight-tied SI (same module for both hops, no new SI parameters).
    """

    def __init__(self, n_channels, n_features=9, hidden_size=64,
                 n_pred_steps=10, init_corr=None):
        super().__init__()
        self.n_channels = n_channels
        self.hidden_size = hidden_size

        self.te = TemporalEncoder(n_features, hidden_size)
        self.si = SpatialInteractionAblatable(
            hidden_size=hidden_size, n_channels=n_channels,
            init_corr=init_corr, use_add=True, use_mul=True,
            use_adaptor=True, learnable_adj=True)
        self.tr = TemporalReadout(hidden_size, n_pred_steps)
        # Learnable gate for second hop (init 0.0 -> sigmoid -> 0.5)
        self.hop2_alpha = nn.Parameter(torch.tensor(0.0))

    def forward(self, x):
        H = self.te(x)              # (B, T, C, d)
        Z1 = self.si(H)             # first hop
        Z2 = self.si(Z1)            # second hop (weight-tied)
        alpha = torch.sigmoid(self.hop2_alpha)
        Z = (1 - alpha) * Z1 + alpha * Z2  # gated blend
        return self.tr(Z)           # (B, T_pred, C)


def make_multihop_amag(monkey, init_corr):
    return AMAG_MultiHop(
        n_channels=MONKEY_CHANNELS[monkey],
        n_features=N_FEATURES,
        hidden_size=HIDDEN_SIZE,
        n_pred_steps=N_PRED_STEPS,
        init_corr=init_corr,
    )


print('AMAG_MultiHop defined.')

In [ ]:
# ── Train Experiment 1 ────────────────────────────────────────────────────────
# Note: 2x SI cost for Affi. If OOM, retry with batch_size=16.
try:
    exp1_results = run_experiment('exp1_multihop', make_multihop_amag)
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print('\nOOM on default batch size! Retrying with batch_size=16...')
        torch.cuda.empty_cache()
        exp1_results = run_experiment('exp1_multihop', make_multihop_amag,
                                     batch_size=16)
    else:
        raise

In [ ]:
# ── Experiment 1: Results ─────────────────────────────────────────────────────
print_comparison_table(['baseline', 'exp1_multihop'], title='Exp 1: Multi-Hop SI')

# Inspect the learned gate value
for monkey in MONKEYS:
    ckpt_path = os.path.join(DRIVE_RESULTS, 'exp1_multihop', monkey, 'checkpoints', 'best.pth')
    if os.path.exists(ckpt_path):
        state = torch.load(ckpt_path, weights_only=False, map_location='cpu')
        alpha_raw = state['model_state_dict']['hop2_alpha'].item()
        alpha = 1.0 / (1.0 + np.exp(-alpha_raw))  # sigmoid
        print(f'  {monkey}: hop2_alpha (raw)={alpha_raw:.4f}, sigmoid(alpha)={alpha:.4f}')
        print(f'    Interpretation: {alpha*100:.1f}% second hop / {(1-alpha)*100:.1f}% first hop')

In [ ]:
# ── Experiment 1: Diagnostic Plots ────────────────────────────────────────────
plot_experiment_comparison(['baseline', 'exp1_multihop'], title_suffix='(Exp 1)')
plot_learning_curves(['baseline', 'exp1_multihop'], title_suffix='(Exp 1)')

---
# Section 4: Experiment 2 -- Delta Input for Temporal Readout

## Problem

The TR GRU receives `z_t` at each timestep, where `z_t = beta1*h_t + beta2*FC(a_t) + beta3*FC(m_t)`. The `h_t` component already encodes the full temporal history up to `t` (it is the TE GRU's hidden state). So feeding `[z_0, z_1, ..., z_9]` to the TR GRU gives it representations with **heavily overlapping temporal content** -- `z_9`'s `h_9` component already summarizes the entire sequence.

The TR GRU wastes capacity re-encoding temporal information that `h_9` has already summarized.

## Change

1. Feed the TR GRU the **spatial delta** `Delta_t = z_t - h_t` instead of `z_t`
2. Initialize the TR GRU's hidden state with `h_9` (the TE's final temporal summary) instead of zeros

This way the TR starts with the full temporal summary and only needs to integrate the spatial corrections over time.

## Hypothesis

Reduces redundant temporal re-encoding. The TR can allocate its full capacity to processing the sequence of spatial corrections, potentially producing better predictions -- especially at later timesteps where the spatial signal has had more time to accumulate.

In [ ]:
# ── Experiment 2: Delta TR Model ─────────────────────────────────────────────

class TemporalReadoutWithH0(nn.Module):
    """
    Modified Temporal Readout that accepts an initial hidden state h0.
    Used with delta input: Z_input = Z_si - H_te (pure spatial correction).
    """

    def __init__(self, hidden_size, n_pred_steps=10):
        super().__init__()
        self.hidden_size = hidden_size
        self.n_pred_steps = n_pred_steps
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, n_pred_steps)

    def forward(self, Z, h0=None):
        """
        Args:
            Z: (B, T, C, d) -- spatial delta or regular input
            h0: (B, C, d) -- initial hidden state (e.g., TE's final state)
        Returns:
            pred: (B, T_pred, C)
        """
        B, T, C, d = Z.shape
        Z_bc = Z.permute(0, 2, 1, 3).reshape(B * C, T, d)

        if h0 is not None:
            h0_bc = h0.reshape(B * C, d).unsqueeze(0).contiguous()  # (1, B*C, d)
            _, h_n = self.gru(Z_bc, h0_bc)
        else:
            _, h_n = self.gru(Z_bc)

        h_last = h_n.squeeze(0)  # (B*C, d)
        pred_bc = self.fc(h_last)  # (B*C, T_pred)
        return pred_bc.reshape(B, C, self.n_pred_steps).permute(0, 2, 1)


class AMAG_DeltaTR(ForecastingModel):
    """
    AMAG with delta input for TR and h0 initialization.
    Forward: H -> Z = SI(H) -> Delta = Z - H -> TR(Delta, h0=H[:,-1]) -> pred
    """

    def __init__(self, n_channels, n_features=9, hidden_size=64,
                 n_pred_steps=10, init_corr=None):
        super().__init__()
        self.n_channels = n_channels
        self.hidden_size = hidden_size

        self.te = TemporalEncoder(n_features, hidden_size)
        self.si = SpatialInteractionAblatable(
            hidden_size=hidden_size, n_channels=n_channels,
            init_corr=init_corr, use_add=True, use_mul=True,
            use_adaptor=True, learnable_adj=True)
        self.tr = TemporalReadoutWithH0(hidden_size, n_pred_steps)

    def forward(self, x):
        H = self.te(x)              # (B, T, C, d)
        Z = self.si(H)              # (B, T, C, d)
        Delta = Z - H               # (B, T, C, d) spatial correction
        h_last = H[:, -1, :, :]     # (B, C, d) TE's final temporal summary
        return self.tr(Delta, h0=h_last)  # (B, T_pred, C)


def make_delta_tr_amag(monkey, init_corr):
    return AMAG_DeltaTR(
        n_channels=MONKEY_CHANNELS[monkey],
        n_features=N_FEATURES,
        hidden_size=HIDDEN_SIZE,
        n_pred_steps=N_PRED_STEPS,
        init_corr=init_corr,
    )


print('AMAG_DeltaTR defined.')

In [ ]:
# ── Train Experiment 2 ────────────────────────────────────────────────────────
exp2_results = run_experiment('exp2_delta_tr', make_delta_tr_amag)

In [ ]:
# ── Experiment 2: Results ─────────────────────────────────────────────────────
print_comparison_table(['baseline', 'exp2_delta_tr'], title='Exp 2: Delta TR')

In [ ]:
# ── Experiment 2: Diagnostic Plots ────────────────────────────────────────────
plot_experiment_comparison(['baseline', 'exp2_delta_tr'], title_suffix='(Exp 2)')
plot_learning_curves(['baseline', 'exp2_delta_tr'], title_suffix='(Exp 2)')

---
# Section 5: Experiment 3 -- Interleaved TE-SI + Auxiliary Next-Step Prediction Loss

## Problem 1: Temporal Isolation of TE

The TE GRU processes each channel in **complete temporal isolation**. At timestep t, channel v's hidden state `h_t^v` encodes zero information about its neighbors' activity. Spatial interaction only happens AFTER the TE has finished processing all 10 timesteps -- by which point the temporal encoding is already fixed and cannot be influenced by spatial context.

In the actual brain, spatial interactions shape each channel's dynamics **continuously at every timestep**. M1's response at t=5 is influenced by PM's activity at t=4, which the TE GRU never sees.

## Problem 2: Weak SI Training Signal

The SI module's only gradient comes from the final 10-step-ahead prediction loss, flowing backward through the entire TR GRU. This is a long gradient path that provides a weak, indirect signal for learning spatial interactions.

## Changes

**Architecture**: Interleave TE and SI. At each timestep:
1. GRU step: `h_t = GRUCell(h_{t-1}, x_t)`
2. SI step: `z_t = SI(h_t for all channels)`
3. Gated residual: `h_t <- h_t + alpha * tanh(W_gate * (z_t - h_t))`

The GRU hidden state now accumulates spatial context across time.

**Auxiliary loss**: `L_aux = mean over t=0..8 of ||Decoder(z_t) - x_{t+1}||^2`

This gives the SI module a **direct per-timestep training signal**: did your spatial enrichment help predict the next observation? This is motivated by predictive coding theory -- the cortex continuously predicts its own next state, and learning is driven by prediction errors.

Total loss: `L = L_forecast + 0.1 * L_aux`

## Hypothesis

- Interleaving gives the temporal encoder spatial awareness, producing richer hidden states
- The auxiliary loss anchors the SI to learn physically meaningful spatial patterns
- Both changes together should improve MSE, especially at later prediction timesteps

In [ ]:
# ── Experiment 3: Interleaved TE-SI + Aux Loss Model ──────────────────────────

class InterleavedTemporalEncoder(nn.Module):
    """
    TE that interleaves spatial interaction at each timestep.
    At each t:
      1. h_t = GRUCell(h_{t-1}, x_t)           -- temporal step
      2. z_t = SI(h_t for all channels)          -- spatial step
      3. h_t <- h_t + alpha*tanh(W*(z_t - h_t))  -- gated residual
    """

    def __init__(self, input_size, hidden_size, n_channels, init_corr=None):
        super().__init__()
        self.hidden_size = hidden_size
        self.n_channels = n_channels

        # GRU cell for step-by-step unrolling
        self.gru_cell = nn.GRUCell(input_size, hidden_size)

        # Spatial interaction module (same architecture as baseline)
        self.si = SpatialInteractionAblatable(
            hidden_size=hidden_size, n_channels=n_channels,
            init_corr=init_corr, use_add=True, use_mul=True,
            use_adaptor=True, learnable_adj=True)

        # Gated residual: controls how much spatial info feeds back
        self.W_gate = nn.Linear(hidden_size, hidden_size, bias=False)
        self.alpha = nn.Parameter(torch.tensor(0.0))  # sigmoid(0) = 0.5

    def forward(self, x):
        """
        Args:
            x: (B, T, C, n_feat)
        Returns:
            H: (B, T, C, d) -- interleaved hidden states (for TR)
            Z_all: (B, T, C, d) -- SI outputs at each step (for aux loss)
        """
        B, T, C, n_feat = x.shape
        device = x.device

        h = torch.zeros(B * C, self.hidden_size, device=device)
        H_list = []
        Z_list = []

        alpha = torch.sigmoid(self.alpha)

        for t in range(T):
            x_t = x[:, t, :, :]  # (B, C, n_feat)
            x_bc = x_t.reshape(B * C, n_feat)  # (B*C, n_feat)

            # Step 1: GRU temporal step
            h = self.gru_cell(x_bc, h)  # (B*C, d)
            h_shaped = h.reshape(B, C, self.hidden_size)  # (B, C, d)

            # Step 2: SI spatial step (single timestep)
            h_for_si = h_shaped.unsqueeze(1)  # (B, 1, C, d)
            z_for_si = self.si(h_for_si)      # (B, 1, C, d)
            z_t = z_for_si.squeeze(1)         # (B, C, d)
            Z_list.append(z_t)

            # Step 3: Gated residual update
            spatial_delta = z_t - h_shaped                  # (B, C, d)
            gate = alpha * torch.tanh(self.W_gate(spatial_delta))  # (B, C, d)
            h_updated = h_shaped + gate                     # (B, C, d)

            H_list.append(h_updated)
            # Feed updated h back into GRU for next timestep
            h = h_updated.reshape(B * C, self.hidden_size)

        H = torch.stack(H_list, dim=1)      # (B, T, C, d)
        Z_all = torch.stack(Z_list, dim=1)  # (B, T, C, d)
        return H, Z_all


class AMAG_Interleaved(ForecastingModel):
    """
    AMAG with interleaved TE-SI and auxiliary next-step prediction loss.

    During training: returns (pred, aux_loss)
    During eval: returns pred only
    """

    def __init__(self, n_channels, n_features=9, hidden_size=64,
                 n_pred_steps=10, init_corr=None):
        super().__init__()
        self.n_channels = n_channels
        self.hidden_size = hidden_size
        self.n_features = n_features

        self.te_si = InterleavedTemporalEncoder(
            n_features, hidden_size, n_channels, init_corr)
        self.tr = TemporalReadout(hidden_size, n_pred_steps)

        # Auxiliary decoder: predict next timestep's all 9 features from z_t
        self.aux_decoder = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        """
        Args:
            x: (B, T, C, n_feat)
        Returns:
            During training: (pred, aux_loss)
            During eval: pred
        """
        H, Z_all = self.te_si(x)  # both (B, T, C, d)
        pred = self.tr(H)          # (B, T_pred, C)

        if self.training:
            # Auxiliary loss: predict x_{t+1} from z_t for t=0..T-2
            z_for_aux = Z_all[:, :-1, :, :]  # (B, T-1, C, d) = z_0..z_{T-2}
            x_targets = x[:, 1:, :, :]       # (B, T-1, C, n_feat) = x_1..x_{T-1}
            x_pred = self.aux_decoder(z_for_aux)  # (B, T-1, C, n_feat)
            aux_loss = nn.functional.mse_loss(x_pred, x_targets)
            return pred, aux_loss
        else:
            return pred


def make_interleaved_amag(monkey, init_corr):
    return AMAG_Interleaved(
        n_channels=MONKEY_CHANNELS[monkey],
        n_features=N_FEATURES,
        hidden_size=HIDDEN_SIZE,
        n_pred_steps=N_PRED_STEPS,
        init_corr=init_corr,
    )


print('AMAG_Interleaved defined.')

In [ ]:
# ── Train Experiment 3 ────────────────────────────────────────────────────────
exp3_results = run_experiment(
    'exp3_interleaved',
    make_interleaved_amag,
    trainer_class=AuxLossTrainer,
    trainer_kwargs={'aux_weight': 0.1},
)

In [ ]:
# ── Experiment 3: Results ─────────────────────────────────────────────────────
print_comparison_table(['baseline', 'exp3_interleaved'], title='Exp 3: Interleaved + Aux Loss')

# Inspect the learned gate value
for monkey in MONKEYS:
    ckpt_path = os.path.join(DRIVE_RESULTS, 'exp3_interleaved', monkey, 'checkpoints', 'best.pth')
    if os.path.exists(ckpt_path):
        state = torch.load(ckpt_path, weights_only=False, map_location='cpu')
        alpha_raw = state['model_state_dict']['te_si.alpha'].item()
        alpha = 1.0 / (1.0 + np.exp(-alpha_raw))
        print(f'  {monkey}: interleave alpha (raw)={alpha_raw:.4f}, sigmoid={alpha:.4f}')
        print(f'    Interpretation: {alpha*100:.1f}% spatial feedback into temporal state')

In [ ]:
# ── Experiment 3: Diagnostic Plots ────────────────────────────────────────────
plot_experiment_comparison(['baseline', 'exp3_interleaved'], title_suffix='(Exp 3)')
plot_learning_curves(['baseline', 'exp3_interleaved'], title_suffix='(Exp 3)')

---
# Section 6: Final Comparison

Master comparison across all four models (baseline + 3 experiments) on both monkeys.

In [ ]:
# ── Master Comparison Table ───────────────────────────────────────────────────
ALL_EXPERIMENTS = ['baseline', 'exp1_multihop', 'exp2_delta_tr', 'exp3_interleaved']
print_comparison_table(ALL_EXPERIMENTS, title='FINAL COMPARISON -- All Experiments')

In [ ]:
# ── Per-Timestep MSE: All Models ──────────────────────────────────────────────
plot_experiment_comparison(ALL_EXPERIMENTS, title_suffix='(All Models)')

In [ ]:
# ── Per-Channel MSE Scatter: Each Experiment vs Baseline ──────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, monkey in enumerate(MONKEYS):
    base_key = ('baseline', monkey)
    if base_key not in ALL_RESULTS:
        continue
    base_ch_mse = np.array(ALL_RESULTS[base_key]['per_channel_mse'])

    for j, exp_name in enumerate(['exp1_multihop', 'exp2_delta_tr', 'exp3_interleaved']):
        exp_key = (exp_name, monkey)
        if exp_key not in ALL_RESULTS:
            continue

        ax = axes[idx][j]
        exp_ch_mse = np.array(ALL_RESULTS[exp_key]['per_channel_mse'])

        ax.scatter(base_ch_mse, exp_ch_mse, alpha=0.4, s=10)
        lims = [0, max(base_ch_mse.max(), exp_ch_mse.max()) * 1.05]
        ax.plot(lims, lims, 'r--', alpha=0.5, label='y=x')
        ax.set_xlabel('Baseline per-ch MSE')
        ax.set_ylabel(f'{exp_name} per-ch MSE')
        ax.set_title(f'{monkey.capitalize()} -- {exp_name}')
        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.legend(fontsize=8)

        # Count improved channels
        n_improved = (exp_ch_mse < base_ch_mse).sum()
        n_total = len(base_ch_mse)
        ax.text(0.05, 0.95, f'{n_improved}/{n_total} improved',
                transform=ax.transAxes, fontsize=9, va='top')

plt.tight_layout()
save_path = os.path.join(DRIVE_RESULTS, 'per_channel_scatter.png')
plt.savefig(save_path, dpi=150)
plt.show()
print(f'Saved: {save_path}')

In [ ]:
# ── Improvement Summary Bar Chart ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

exp_names_short = ['Multi-Hop\n(Exp 1)', 'Delta TR\n(Exp 2)', 'Interleaved\n(Exp 3)']
exp_keys = ['exp1_multihop', 'exp2_delta_tr', 'exp3_interleaved']

x = np.arange(len(exp_keys))
width = 0.35

for idx, monkey in enumerate(MONKEYS):
    base = ALL_RESULTS.get(('baseline', monkey))
    if base is None:
        continue
    pcts = []
    for ek in exp_keys:
        exp = ALL_RESULTS.get((ek, monkey))
        if exp:
            pct = (exp['mse'] - base['mse']) / base['mse'] * 100
            pcts.append(pct)
        else:
            pcts.append(0)
    offset = -width/2 + idx * width
    bars = ax.bar(x + offset, pcts, width, label=monkey.capitalize())
    for bar, pct in zip(bars, pcts):
        color = 'green' if pct < 0 else 'red'
        bar.set_color(color if idx == 0 else ('darkgreen' if pct < 0 else 'darkred'))
        bar.set_alpha(0.7 if idx == 0 else 0.5)

ax.set_xticks(x)
ax.set_xticklabels(exp_names_short)
ax.set_ylabel('% MSE Change vs Baseline')
ax.set_title('Experiment Results: MSE Change vs Baseline (negative = better)')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
save_path = os.path.join(DRIVE_RESULTS, 'improvement_summary.png')
plt.savefig(save_path, dpi=150)
plt.show()
print(f'Saved: {save_path}')

In [ ]:
# ── Save All Results to Drive ─────────────────────────────────────────────────

def convert_for_json(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, dict):
        return {k: convert_for_json(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [convert_for_json(v) for v in obj]
    return obj

# Convert ALL_RESULTS to serializable format
results_serializable = {}
for (exp_name, monkey), data in ALL_RESULTS.items():
    key = f'{exp_name}/{monkey}'
    # Exclude history (too large) -- it's already saved per-experiment
    results_serializable[key] = {
        k: convert_for_json(v) for k, v in data.items() if k != 'history'
    }

results_path = os.path.join(DRIVE_RESULTS, 'all_results.json')
with open(results_path, 'w') as f:
    json.dump(results_serializable, f, indent=2)

print(f'All results saved to: {results_path}')
print(f'\nPlots saved to: {DRIVE_RESULTS}/')
print('\nExperiment complete!')

---
# Summary

## What was tested

| Experiment | Architectural Change | New Params |
|-----------|---------------------|------------|
| **Baseline** | Standard AMAG: TE -> SI -> TR | 0 |
| **Exp 1: Multi-Hop** | Two SI passes (2-hop propagation) | 1 (gate scalar) |
| **Exp 2: Delta TR** | Feed spatial delta to TR, init from TE's final state | 0 |
| **Exp 3: Interleaved** | Interleave TE+SI per timestep + auxiliary next-step prediction loss | ~4,673 |

## Interpreting Results

- **MSE improvement**: Lower is better. A >1% relative reduction is meaningful.
- **Affi vs Beignet delta**: Exp 1 should help Affi more (multi-region). If it helps Beignet equally, the second hop is likely just regularizing, not capturing inter-regional paths.
- **Per-timestep MSE**: Changes in the error curve shape reveal WHAT the model learned -- whether it improved short-horizon or long-horizon predictions.
- **Per-channel scatter**: Points below the diagonal are channels that improved. Clusters of improvement may correspond to specific brain regions.
- **Learned gate values**: alpha near 0 means the change is being ignored; alpha near 1 means it's fully adopted.

## Next Steps

Based on which experiments improve MSE, consider:
- Combining successful modifications (e.g., Multi-Hop + Delta TR)
- Tuning aux_weight for Exp 3 (try 0.01, 0.05, 0.2)
- Adding L1 sparsity penalty on adjacency matrices
- Testing on the held-out test set for final competition submission